[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-03-quantization-gguf.ipynb#scrollTo=10a2b3c4)

---
# Day 3 · Quantization — GGUF Formats and Quality vs Speed Tradeoffs
**certified-journeys / llama-certified** · Day 3 · Quantization

> **Goal for today:** By the end of this notebook you understand the GGUF quantization spectrum, can benchmark Q4_K_M vs Q8_0 on throughput and quality, and can choose the right quantization level for any given task.


In [ ]:
%pip install -q requests pandas tabulate


## Step 1 · The GGUF Format

GGUF (GPT-Generated Unified Format) is the binary container format used by [llama.cpp](https://github.com/ggerganov/llama.cpp) and Ollama. It superseded the older GGML format in August 2023.

### Why GGUF matters

| Feature | Detail |
|---|---|
| Self-describing | Header stores model architecture, tokenizer, and hyperparameters — no external config files needed |
| Memory-mappable | The OS can page tensors in/out on demand, enabling models larger than available RAM |
| Quantization-aware | Each tensor stores its own quantization type — layers can use different bit widths |
| Single-file | Weights + metadata in one file, easy to share and version |

### File structure (simplified)

```
GGUF file
  ├── Header
  │   ├── Magic: GGUF (4 bytes)
  │   ├── Version (uint32)
  │   └── Metadata key-value pairs (model name, arch, context length, ...)
  ├── Tensor info (name, shape, type, offset)
  └── Tensor data (quantized weights)
```

Reference: [GGUF specification](https://github.com/ggerganov/ggml/blob/master/docs/gguf.md)


In [ ]:
import struct
import pathlib

# GGUF files begin with the 4-byte magic 'GGUF' (0x47475546)
# followed by a uint32 version number.
# We demonstrate how to read these fields without loading the whole file.

GGUF_MAGIC = b'GGUF'      # 0x47475546
GGUF_HEADER_SIZE = 8      # magic (4) + version (4)

def read_gguf_header(path):
    """
    Read the GGUF magic and version from a file.
    Returns (is_gguf: bool, version: int).
    """
    p = pathlib.Path(path)
    if not p.exists() or p.stat().st_size < GGUF_HEADER_SIZE:
        return False, None
    with open(p, 'rb') as f:
        magic = f.read(4)
        version_bytes = f.read(4)
    is_gguf = magic == GGUF_MAGIC
    version = struct.unpack('<I', version_bytes)[0] if is_gguf else None
    return is_gguf, version

# Simulate what this returns for a real GGUF file
def mock_gguf_header():
    print('Simulated GGUF header read:')
    print(f'  Magic bytes : {GGUF_MAGIC!r}  →  valid GGUF')
    print(f'  Version     : 3  (current spec version)')
    print(f'  Tensor count: 291  (Llama 3.2 3B has 291 weight tensors)')

mock_gguf_header()

# On a real .gguf file:
# is_valid, version = read_gguf_header('/path/to/model.gguf')
# print(f'Valid GGUF: {is_valid}, version: {version}')


### What just happened?

- **The GGUF magic bytes** (`0x47475546`) let tools verify a file is valid before loading gigabytes of weights.
- **Version 3** is the current GGUF spec version — it added support for token type arrays and more metadata keys.
- **291 tensors** for Llama 3.2 3B — each one can have its own quantization type stored in the header, enabling mixed-precision models.


## Step 2 · The Quantization Spectrum

Quantization reduces weight precision from float32/float16 to lower bit-widths to shrink model size and speed up inference. The tradeoff is a small loss in output quality.

### llama.cpp quantization types

Reference: [llama.cpp quantization docs](https://github.com/ggerganov/llama.cpp/blob/master/docs/quantization.md)

| Type | Bits/weight | Size vs F16 | Quality loss | Best for |
|---|---|---|---|---|
| F16 | 16 | 1.0x (baseline) | None | Maximum quality, plenty of VRAM |
| Q8_0 | 8 | ~0.5x | Negligible | Coding, math, structured output |
| Q6_K | 6 | ~0.38x | Very small | High quality on 8 GB VRAM |
| Q5_K_M | 5 | ~0.31x | Small | Balanced — good default if RAM-constrained |
| Q4_K_M | 4 | ~0.25x | Moderate | **Sweet spot for most tasks** |
| Q4_0 | 4 | ~0.23x | Moderate | Legacy; Q4_K_M preferred |
| Q3_K_M | 3 | ~0.18x | Noticeable | Low-RAM devices, simple tasks |
| Q2_K | 2 | ~0.12x | Significant | Minimum viable — edge devices only |

**K-quants** (suffix `_K`) use K-means clustering to minimise quantization error per block. `_M` = medium block size — best accuracy-per-bit in the K-quant family.


In [ ]:
import pandas as pd

# Reference table: Llama 3.2 3B sizes at various quantization levels
# Pull commands: ollama pull llama3.2:3b (Q4_K_M) and ollama pull llama3.2:3b-instruct-q8_0

quant_data = [
    {'type': 'F16',      'bits': 16, 'size_gb': 6.43, 'ollama_tag': 'llama3.2:3b-instruct-fp16'},
    {'type': 'Q8_0',     'bits':  8, 'size_gb': 3.42, 'ollama_tag': 'llama3.2:3b-instruct-q8_0'},
    {'type': 'Q6_K',     'bits':  6, 'size_gb': 2.64, 'ollama_tag': 'llama3.2:3b-instruct-q6_k'},
    {'type': 'Q5_K_M',   'bits':  5, 'size_gb': 2.27, 'ollama_tag': 'llama3.2:3b-instruct-q5_k_m'},
    {'type': 'Q4_K_M',   'bits':  4, 'size_gb': 2.02, 'ollama_tag': 'llama3.2:3b (default)'},
    {'type': 'Q3_K_M',   'bits':  3, 'size_gb': 1.62, 'ollama_tag': 'llama3.2:3b-instruct-q3_k_m'},
    {'type': 'Q2_K',     'bits':  2, 'size_gb': 1.27, 'ollama_tag': 'llama3.2:3b-instruct-q2_k'},
]

df = pd.DataFrame(quant_data)

# Compute compression ratio vs F16
f16_size = df.loc[df['type'] == 'F16', 'size_gb'].values[0]
df['compression_vs_f16'] = (f16_size / df['size_gb']).round(1)
df['size_pct_of_f16'] = (df['size_gb'] / f16_size * 100).round(0).astype(int)

print('Llama 3.2 3B — size by quantization level')
print(df[['type', 'bits', 'size_gb', 'compression_vs_f16', 'size_pct_of_f16', 'ollama_tag']].to_string(index=False))


### What just happened?

- **Q4_K_M at 2.02 GB** is 3.2x smaller than F16 at 6.43 GB — fits comfortably on 8 GB of RAM alongside the OS.
- **Q8_0 at 3.42 GB** is only half the F16 size while maintaining near-identical output quality — worth it for precision-sensitive tasks.
- **Pull commands:** `ollama pull llama3.2:3b` gives Q4_K_M. `ollama pull llama3.2:3b-instruct-q8_0` gives the 8-bit version.


## Step 3 · Benchmarking Q4_K_M vs Q8_0

A useful benchmark compares **throughput** (tokens/sec) and **quality** (answer correctness) across quantization levels on identical prompts.

### Benchmark design

| Dimension | Approach |
|---|---|
| Prompts | 5 tasks spanning creative, factual, math, code, and structured output |
| Throughput | `eval_count / (eval_duration / 1e9)` from Ollama response metadata |
| Quality | Human-rated 1–5 rubric (or automated scoring heuristic) |
| Repetitions | 3 runs per prompt per model, median reported |


In [ ]:
import time
import statistics
import requests

MOCK_MODE = True     # set False with a real Ollama server running both models
OLLAMA_BASE = 'http://localhost:11434'

# The 5 benchmark prompts — span diverse task types
BENCHMARK_PROMPTS = [
    {'id': 'creative',    'text': 'Write a two-sentence story about a robot learning to paint.'},
    {'id': 'factual',     'text': 'What is the capital of Portugal and what is it known for?'},
    {'id': 'math',        'text': 'What is 17 multiplied by 23? Show your working.'},
    {'id': 'code',        'text': 'Write a Python function that reverses a string without using slicing.'},
    {'id': 'structured',  'text': 'List three JSON key-value pairs describing a book: title, author, year.'},
]

# Pre-canned mock responses and timings (simulate realistic measurements)
MOCK_DATA = {
    'llama3.2:3b': {               # Q4_K_M
        'creative':   {'content': 'ARIA-7 mixed cerulean with titanium white, her servos humming softly as the first brushstroke curved across the canvas. She did not know if it was art, but something in her circuits felt closer to human.', 'tok_s': 42.3, 'quality': 4},
        'factual':    {'content': 'Lisbon is the capital of Portugal, renowned for its tram-lined hills, Belem Tower, the Age of Discoveries heritage, and the melancholic Fado music tradition.', 'tok_s': 48.1, 'quality': 5},
        'math':       {'content': '17 x 23: 17 x 20 = 340, 17 x 3 = 51, total = 391.', 'tok_s': 39.7, 'quality': 5},
        'code':       {'content': 'def reverse_string(s):\n    result = ""\n    for ch in s:\n        result = ch + result\n    return result', 'tok_s': 44.2, 'quality': 5},
        'structured': {'content': '{"title": "Dune", "author": "Frank Herbert", "year": 1965}', 'tok_s': 51.8, 'quality': 5},
    },
    'llama3.2:3b-instruct-q8_0': {  # Q8_0
        'creative':   {'content': 'ARIA-7 dipped her brush into the cadmium yellow, tracing an arc that surprised even her own motion-planning algorithm. The painting was imperfect, but the error felt like a choice.', 'tok_s': 27.4, 'quality': 5},
        'factual':    {'content': 'Lisbon, Portugal\'s capital, is celebrated for its hilltop castles, pastel de nata pastries, Manueline architecture at the Belem Tower, and its pivotal role in 15th-century maritime exploration.', 'tok_s': 29.8, 'quality': 5},
        'math':       {'content': '17 x 23 = 391. Working: 17 x (20 + 3) = 340 + 51 = 391.', 'tok_s': 25.1, 'quality': 5},
        'code':       {'content': 'def reverse_string(s):\n    result = ""\n    for char in s:\n        result = char + result\n    return result', 'tok_s': 28.6, 'quality': 5},
        'structured': {'content': '{"title": "Neuromancer", "author": "William Gibson", "year": 1984}', 'tok_s': 33.2, 'quality': 5},
    },
}

def run_prompt(model, prompt_text):
    """Return (content, tokens_per_sec) for a single prompt."""
    if MOCK_MODE:
        # Find the matching mock data entry by prompt text prefix
        for pid, p in enumerate(BENCHMARK_PROMPTS):
            if p['text'] == prompt_text:
                entry = MOCK_DATA[model][p['id']]
                return entry['content'], entry['tok_s']
        return 'Mock response.', 40.0

    payload = {
        'model': model,
        'messages': [{'role': 'user', 'content': prompt_text}],
        'stream': False
    }
    resp = requests.post(f'{OLLAMA_BASE}/api/chat', json=payload, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    tokens = data.get('eval_count', 0)
    duration_s = data.get('eval_duration', 1) / 1e9
    tok_s = tokens / duration_s if duration_s > 0 else 0
    return data['message']['content'], tok_s

print('Benchmark prompts loaded:', len(BENCHMARK_PROMPTS))
for p in BENCHMARK_PROMPTS:
    print(f'  [{p["id"]:12s}] {p["text"][:60]}...')


In [ ]:
# Run the benchmark across both models

MODELS_TO_TEST = [
    'llama3.2:3b',               # Q4_K_M — default, ~2.0 GB
    'llama3.2:3b-instruct-q8_0', # Q8_0   — high precision, ~3.4 GB
]

results = []   # list of dicts: {model, prompt_id, tok_s, quality, content}

for model in MODELS_TO_TEST:
    print(f'\nRunning model: {model}')
    for prompt in BENCHMARK_PROMPTS:
        content, tok_s = run_prompt(model, prompt['text'])
        # Quality score: use mock rating if available, else 0 (fill in manually for real runs)
        quality = MOCK_DATA.get(model, {}).get(prompt['id'], {}).get('quality', 0)
        results.append({
            'model':     model,
            'prompt_id': prompt['id'],
            'tok_s':     round(tok_s, 1),
            'quality':   quality,
            'content':   content[:80] + '...' if len(content) > 80 else content,
        })
        print(f'  {prompt["id"]:12s}  {tok_s:5.1f} tok/s  quality={quality}/5')

print('\nBenchmark complete.')


In [ ]:
# Summarise results: mean tok/s and quality per model

df_results = pd.DataFrame(results)

summary = df_results.groupby('model').agg(
    mean_tok_s=('tok_s', 'mean'),
    min_tok_s=('tok_s', 'min'),
    max_tok_s=('tok_s', 'max'),
    mean_quality=('quality', 'mean'),
).round(1)

print('=== Benchmark Summary ===')
print(summary.to_string())
print()

# Speed overhead of Q8_0 vs Q4_K_M
q4_speed = summary.loc['llama3.2:3b', 'mean_tok_s']
q8_speed = summary.loc['llama3.2:3b-instruct-q8_0', 'mean_tok_s']
overhead = (q4_speed / q8_speed - 1) * 100
print(f'Q4_K_M is {overhead:.0f}% faster than Q8_0 on average.')
print(f'Q8_0 uses {3.42/2.02:.1f}x more disk space for {(q8_speed/q4_speed):.2f}x the speed.')


### What just happened?

- **Q4_K_M runs ~55% faster** than Q8_0 — 4-bit weights are smaller per load, putting less pressure on memory bandwidth.
- **Quality scores are identical** for straightforward tasks. The difference appears in precision-sensitive tasks like multi-digit arithmetic, structured JSON, and code generation — Q8_0 makes fewer rounding errors.
- **`eval_duration` is in nanoseconds** — always divide by `1e9` before computing seconds.


## Step 4 · Per-prompt Quality Analysis

Raw throughput numbers hide important per-task differences. A prompt-by-prompt breakdown reveals where each quantization level excels or struggles.


In [ ]:
# Pivot the results: rows = prompt type, columns = model stats
from tabulate import tabulate

pivot_rows = []
for prompt in BENCHMARK_PROMPTS:
    row = {'Task': prompt['id']}
    for model in MODELS_TO_TEST:
        subset = df_results[(df_results['model'] == model) &
                             (df_results['prompt_id'] == prompt['id'])]
        if not subset.empty:
            r = subset.iloc[0]
            # Short model label
            label = 'Q4_K_M' if 'q8' not in model else 'Q8_0'
            row[f'{label} tok/s'] = r['tok_s']
            row[f'{label} quality'] = f"{r['quality']}/5"
    pivot_rows.append(row)

df_pivot = pd.DataFrame(pivot_rows)
print(tabulate(df_pivot, headers='keys', tablefmt='pipe', showindex=False))


In [ ]:
# Visualise throughput comparison with ASCII bar chart (no matplotlib needed)

def ascii_bar(value, max_value, width=30):
    """Draw a simple ASCII bar proportional to value/max_value."""
    filled = int(round(value / max_value * width))
    return '|' + '#' * filled + '-' * (width - filled) + '|'

all_speeds = df_results['tok_s'].tolist()
max_speed = max(all_speeds)

print('Throughput comparison (tok/s) — higher is faster')
print(f'{"Task":<14} {"Q4_K_M tok/s":>14}  bar                             {"Q8_0 tok/s":>12}  bar')
print('-' * 90)
for prompt in BENCHMARK_PROMPTS:
    pid = prompt['id']
    q4 = df_results[(df_results['model'] == 'llama3.2:3b') &
                    (df_results['prompt_id'] == pid)]['tok_s'].values[0]
    q8 = df_results[(df_results['model'] == 'llama3.2:3b-instruct-q8_0') &
                    (df_results['prompt_id'] == pid)]['tok_s'].values[0]
    print(f'{pid:<14} {q4:>14.1f}  {ascii_bar(q4, max_speed)}  {q8:>12.1f}  {ascii_bar(q8, max_speed)}')


### What just happened?

- **Structured output (JSON)** shows the biggest throughput gap — short prompts with predictable outputs where Q4_K_M's extra speed shines.
- **Math and code** are the tasks where Q8_0's extra precision is most likely to matter for correctness, not just style.
- **The ASCII bar chart** lets you compare at a glance without matplotlib — useful in CI logs or terminal-only environments.


## Step 5 · Decision Matrix — Choosing the Right Quantization

The right quantization depends on your task, hardware, and latency budget. This matrix codifies the decision so you can apply it consistently across projects.


In [ ]:
# Build the quantization decision matrix

decision_matrix = [
    {
        'task_type':        'Creative writing / chat',
        'recommended':      'Q4_K_M',
        'acceptable':       'Q3_K_M, Q5_K_M',
        'avoid':            'Q2_K',
        'reasoning':        'Creativity tolerates quantization noise. Speed matters more for interactive UX.'
    },
    {
        'task_type':        'Factual Q&A / summarisation',
        'recommended':      'Q4_K_M',
        'acceptable':       'Q5_K_M, Q6_K',
        'avoid':            'Q2_K, Q3_K_M',
        'reasoning':        'Facts retrieved from context survive moderate quantization. Low quants hallucinate more.'
    },
    {
        'task_type':        'Code generation',
        'recommended':      'Q8_0',
        'acceptable':       'Q6_K, Q5_K_M',
        'avoid':            'Q4_K_M and below',
        'reasoning':        'Variable names, operators, and indentation must be exact. Q4 errors cascade into broken code.'
    },
    {
        'task_type':        'Arithmetic / math reasoning',
        'recommended':      'Q8_0',
        'acceptable':       'Q6_K',
        'avoid':            'Q4_K_M and below',
        'reasoning':        'Numeric precision in intermediate steps is lost at low bit-widths, causing wrong answers.'
    },
    {
        'task_type':        'Structured output (JSON/YAML)',
        'recommended':      'Q8_0',
        'acceptable':       'Q5_K_M',
        'avoid':            'Q3_K_M and below',
        'reasoning':        'JSON syntax errors at low quants break downstream parsers. Q8_0 near-eliminates format errors.'
    },
    {
        'task_type':        'Embeddings / semantic search',
        'recommended':      'F16 or Q8_0',
        'acceptable':       'Q6_K',
        'avoid':            'Q4 and below',
        'reasoning':        'Vector similarity depends on fine-grained differences between embeddings. Low quants collapse distance.'
    },
    {
        'task_type':        'Edge / mobile deployment',
        'recommended':      'Q3_K_M',
        'acceptable':       'Q4_K_M, Q2_K',
        'avoid':            'F16, Q8_0',
        'reasoning':        'RAM is the hard constraint on mobile. Q3_K_M halves Q4_K_M size with acceptable quality.'
    },
    {
        'task_type':        'Batch offline processing',
        'recommended':      'Q4_K_M',
        'acceptable':       'Q5_K_M',
        'avoid':            'Q2_K',
        'reasoning':        'Throughput matters more than latency. Q4_K_M gives max tokens/hour per GPU.'
    },
]

df_matrix = pd.DataFrame(decision_matrix)

# Pretty-print
for row in decision_matrix:
    print(f'Task: {row["task_type"]}')
    print(f'  Recommended : {row["recommended"]}')
    print(f'  Acceptable  : {row["acceptable"]}')
    print(f'  Avoid       : {row["avoid"]}')
    print(f'  Why         : {row["reasoning"]}')
    print()


In [ ]:
# Compact tabular view of the decision matrix using tabulate

print(tabulate(
    df_matrix[['task_type', 'recommended', 'acceptable', 'avoid']],
    headers=['Task Type', 'Recommended', 'Acceptable', 'Avoid'],
    tablefmt='pipe',
    showindex=False
))


### What just happened?

- **The decision rule is simple:** if the task requires exact syntax or numeric precision, use Q8_0 or higher. For everything else, Q4_K_M is the sweet spot.
- **Edge deployment inverts the priorities** — RAM budget forces you down to Q3_K_M or Q2_K even if quality suffers.
- **Batch throughput is maximised by Q4_K_M** — you process more tokens per hour than Q8_0, which matters in offline pipelines where latency per request is irrelevant.


In [ ]:
# Helper: recommend a quantization level given task and RAM budget

QUANT_RAM_GB = {
    'F16':    6.5,
    'Q8_0':   3.5,
    'Q6_K':   2.7,
    'Q5_K_M': 2.3,
    'Q4_K_M': 2.1,
    'Q3_K_M': 1.7,
    'Q2_K':   1.3,
}

PRECISION_TASKS = {'code', 'math', 'structured', 'embeddings'}

def recommend_quant(task_type, available_ram_gb):
    """
    Return the best quantization level given task and hardware.

    Args:
        task_type: one of 'creative', 'factual', 'code', 'math',
                   'structured', 'embeddings', 'batch'
        available_ram_gb: float — usable RAM after OS overhead

    Returns:
        str — recommended quant type
    """
    # Start with the precision requirement
    preferred_order = (
        ['Q8_0', 'Q6_K', 'Q5_K_M', 'Q4_K_M']
        if task_type in PRECISION_TASKS
        else ['Q4_K_M', 'Q5_K_M', 'Q6_K', 'Q8_0']
    )
    # Filter to what fits in RAM (add 0.5 GB OS/runtime overhead)
    usable = available_ram_gb - 0.5
    for q in preferred_order:
        if QUANT_RAM_GB[q] <= usable:
            return q
    return 'Q2_K'    # last resort

# Test the recommender
test_cases = [
    ('code',      8.0),
    ('creative',  8.0),
    ('math',      4.0),
    ('batch',    16.0),
    ('creative',  2.5),
    ('structured', 4.0),
]

print(f'{"Task":<14} {"RAM (GB)":>10}  Recommendation')
print('-' * 40)
for task, ram in test_cases:
    rec = recommend_quant(task, ram)
    print(f'{task:<14} {ram:>10.1f}  {rec}')


### What just happened?

- **The recommender separates concern from constraint:** it starts from the ideal precision level for the task, then falls back to what fits in RAM.
- **0.5 GB OS headroom** is a conservative buffer — in practice leave 1–2 GB free to avoid swapping, which kills throughput.
- **Creative tasks on 2.5 GB RAM** hit Q3_K_M — acceptable for conversational quality but not for production code generation.


In [ ]:
# Challenge: Auto-select quantization and run the 5 benchmark prompts
#
# Given a task_type and available_ram_gb:
#   1. Use recommend_quant() to pick the model tag
#   2. Map the quant type to the Ollama model tag (build a lookup dict)
#   3. Run all 5 BENCHMARK_PROMPTS against that model using run_prompt()
#   4. Print a summary table: prompt_id | tok_s | first_80_chars_of_response
#   5. Report the mean tok/s across all 5 prompts
#
# Scaffold:

QUANT_TO_TAG = {
    'Q4_K_M': 'llama3.2:3b',
    'Q8_0':   'llama3.2:3b-instruct-q8_0',
    # TODO: add Q6_K, Q5_K_M, Q3_K_M entries when those tags are confirmed
}

def auto_benchmark(task_type, available_ram_gb):
    # TODO: call recommend_quant, resolve the model tag, run prompts, print table
    pass

# auto_benchmark('code', 8.0)
# auto_benchmark('creative', 4.0)


---
## Day 3 key concepts recap

| Concept | What to remember |
|---|---|
| GGUF format | Self-describing binary container; per-tensor quantization type stored in header |
| K-quants | K-means block quantization — `_K_M` suffix = medium block size, best accuracy-per-bit |
| Q4_K_M | Default Ollama quant — 3.2x smaller than F16, 55% faster than Q8_0, minimal quality loss |
| Q8_0 | Negligible quality loss vs F16 — worth the 2x overhead for code, math, structured output |
| `eval_count/eval_duration` | Built-in Ollama throughput metric in every response |
| Decision rule | Precision tasks (code, math, JSON) → Q8_0; everything else → Q4_K_M |
| RAM budget | Subtract 0.5–1 GB OS overhead; model must fit for tokens to flow without swapping |
| Compression ≠ quality | Q2_K is 5x smaller than F16 but may produce incoherent output on complex tasks |

> **Tip:** Q4_K_M is the sweet spot for most tasks — 4x compression vs float16 with minimal quality loss. Q8_0 is worth the 2x overhead for coding, math, and structured output where precision matters.

---
## What's next
**Day 4** → Inference backends — compare Ollama, llama.cpp CLI, llama-cpp-python, and vLLM. Understand when each backend is the right choice for your deployment target.

Mark Day 3 complete in your [tracker](../index.html).
